In [1]:
!pip install -q liac-arff pandas || pip install -q liac-arff pandas --break-system-packages

  Preparing metadata (setup.py) ... done


# 04. NASA/PROMISE Data Loading (QA/Defect Domain Substitute): NASA/PROMISE Defect Datasets

Loads 8 real, peer-reviewed NASA/PROMISE software defect datasets (CM1, KC1, JM1, PC1,
PC3, PC4, KC3, MW1), originally released by Shepperd, Song, Sun, and Mair (2014) and
mirrored for research reuse at
[github.com/klainfo/NASADefectDataset](https://github.com/klainfo/NASADefectDataset).

Source ARFF files are in `../data/raw/nasa_promise_arff/`.


In [2]:
import os as _os_setup
for _d in ["../data/raw", "../data/cleaned", "../figures"]:
    _os_setup.makedirs(_d, exist_ok=True)

import arff
import pandas as pd
import os, urllib.request

DATA_DIR = "../data/raw/nasa_promise_arff"
FILES = ['CM1.arff','KC1.arff','JM1.arff','PC1.arff','PC3.arff','PC4.arff','KC3.arff','MW1.arff']

# If the ARFF files aren't already present locally (e.g., this notebook was uploaded to
# Colab on its own, without the rest of the repo), download the real files directly from
# their original GitHub source rather than relying on any local copy.
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/klainfo/NASADefectDataset/master/CleanedData/MDP/D%27%27"

os.makedirs(DATA_DIR, exist_ok=True)
for fname in FILES:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        url = f"{GITHUB_RAW_BASE}/{fname}"
        print(f"Downloading real source file: {fname} ...")
        req = urllib.request.Request(url, headers={"User-Agent": "qm640-capstone"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            with open(fpath, "wb") as out:
                out.write(resp.read())
print("All 8 real NASA/PROMISE ARFF files are present locally.")

COMMON = ['BRANCH_COUNT', 'CYCLOMATIC_COMPLEXITY', 'DESIGN_COMPLEXITY', 'ESSENTIAL_COMPLEXITY',
          'HALSTEAD_CONTENT', 'HALSTEAD_DIFFICULTY', 'HALSTEAD_EFFORT', 'HALSTEAD_ERROR_EST',
          'HALSTEAD_LENGTH', 'HALSTEAD_LEVEL', 'HALSTEAD_PROG_TIME', 'HALSTEAD_VOLUME',
          'LOC_BLANK', 'LOC_CODE_AND_COMMENT', 'LOC_COMMENTS', 'LOC_EXECUTABLE', 'LOC_TOTAL',
          'NUM_OPERANDS', 'NUM_OPERATORS', 'NUM_UNIQUE_OPERANDS', 'NUM_UNIQUE_OPERATORS']

print(f"Loading {len(FILES)} real NASA/PROMISE project files...")

All 8 real NASA/PROMISE ARFF files are present locally.
Loading 8 real NASA/PROMISE project files...


## Load and standardize each project file

In [3]:
def load_one(fname):
    path = os.path.join(DATA_DIR, fname)
    with open(path) as fh:
        d = arff.load(fh)
    cols = [a[0] for a in d['attributes']]
    df = pd.DataFrame(d['data'], columns=cols)
    # normalize label column name (JM1 uses 'label' instead of 'Defective')
    if 'Defective' in df.columns:
        label_col = 'Defective'
    elif 'label' in df.columns:
        label_col = 'label'
    else:
        raise ValueError(f"No label column found in {fname}: {cols}")
    df = df.rename(columns={label_col: 'Defective'})
    df['project'] = fname.replace('.arff', '')
    keep = COMMON + ['Defective', 'project']
    return df[keep]

frames = [load_one(f) for f in FILES]
raw = pd.concat(frames, ignore_index=True)
raw['Defective_bin'] = (raw['Defective'] == 'Y').astype(int)

## Summary of the combined real dataset

In [4]:
print("Total rows:", len(raw))
print("\nRows per project:")
print(raw.groupby('project').size())
print("\nDefect rate per project:")
print((raw.groupby('project')['Defective_bin'].mean() * 100).round(2))
print(f"\nOverall defect rate: {raw['Defective_bin'].mean()*100:.2f}%")

Total rows: 12655

Rows per project:
project
CM1     327
JM1    7720
KC1    1162
KC3     194
MW1     250
PC1     679
PC3    1053
PC4    1270
dtype: int64

Defect rate per project:
project
CM1    12.84
JM1    20.88
KC1    25.30
KC3    18.56
MW1    10.00
PC1     8.10
PC3    12.35
PC4    13.86
Name: Defective_bin, dtype: float64

Overall defect rate: 18.73%


In [5]:
raw.to_csv("../data/raw/nasa_promise_combined_raw.csv", index=False)
print(f"Saved combined_raw.csv, shape: {raw.shape}")

Saved combined_raw.csv, shape: (12655, 24)
